In [ ]:
!pip install sumy parsel pycountry tensorflow_hub nltk transformers tensorflow

In [ ]:
import gc
import pandas as pd
from transformers import pipeline
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from tqdm import tqdm
from parsel import Selector
import time
import tensorflow as tf
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow_hub as hub
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import re
import pycountry
import string
import tensorflow_hub as hub
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras import layers, models, callbacks

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# 1. Data Collection

In [ ]:
!git clone -n --depth=1 --filter=tree:0 \
  https://github.com/CoronaNetDataScience/corona_tscs

!cd corona_tscs && git sparse-checkout set --no-cone /data/CoronaNet/data_country/coronanet_release && git checkout

In [ ]:
dfs = []
for csv in os.listdir('corona_tscs/data/CoronaNet/data_country/coronanet_release'):
    # read each csv into a separate dataframe
    df = pd.read_csv(os.path.join('corona_tscs/data/CoronaNet/data_country/coronanet_release', csv), low_memory=False)
    dfs.append(df)

big_frame = pd.concat(dfs, ignore_index=True)

In [ ]:
big_frame.shape

In [ ]:
big_frame.columns

In [ ]:
big_frame.head(10)

In [ ]:
gc.collect()

del(dfs)

In [ ]:
df = big_frame[["description", "type"]].drop_duplicates(subset=['description']).reset_index(drop=True)

In [ ]:
df

In [ ]:
df['type'].value_counts()

## 2. Preprocess Data

In [ ]:
# Combine all preprocessing steps
def preprocess_text(text):

    # 1. Convert text to lowercase
    def to_lower(text):
        return text.lower()

    # 2. Remove punctuation and numbers
    def remove_punctuation_numbers(text):
        text = re.sub(r'\d+', '', text)  # Remove numbers
        text = re.sub(r'\b(\d+)(st|nd|rd|th)\b', '', text)  # Remove Ordered Numbering
        return text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation

    # 3. Tokenize text into words
    def tokenize(text):
        return nltk.word_tokenize(text)

    # 4. Remove stopwords (common words with little meaning)
    stop_words = set(stopwords.words('english'))
    def remove_stopwords(tokens):
        return [word for word in tokens if word not in stop_words]

    # 5. Apply lemmatization to reduce words to their base (dictionary) form
    lemmatizer = WordNetLemmatizer()
    def apply_lemmatization(tokens):
        return [lemmatizer.lemmatize(word) for word in tokens]

    text = to_lower(text)
    text = remove_punctuation_numbers(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = apply_lemmatization(tokens)
    return " ".join(tokens)

# 6- Apply Preprocessing
df["description_processed"] = df["description"].apply(preprocess_text)

In [ ]:
stop_words = list(stopwords.words('english'))

# include country names in stopwords
country_text = []
for text in df["description_processed"].tolist():
    for c in pycountry.countries:
        if c.name.lower() in text:
            text = re.sub(c.name.lower(), '', text)
    country_text.append(text)

df["description_2"] = country_text

for stop_word in stop_words:
    regex_stopword = r"\b" + stop_word + r"\b"
    df['description_processed'] = df['description_processed'].str.replace(regex_stopword, '')

In [ ]:
df

In [ ]:
list_columns = ["description_2", "type"]
df2 = df[list_columns]

df2 = df2.rename(columns={'description_2': 'description'})

# Modelling

#### Label encoding

In [ ]:
category_codes = {
    'Anti-Disinformation Measures': 0,
    'Hygiene': 1,
    'Curfew': 2,
    'Closure and Regulation of Schools': 3,
    'Declaration of Emergency': 4,
    'External Border Restrictions': 5,
    'Health Monitoring': 6,
    'Health Resources': 7,
    'Health Testing': 8,
    'Internal Border Restrictions': 9,
    'Lockdown': 10,
    'New Task Force, Bureau or Administrative Configuration': 11,
    'COVID-19 Vaccines': 12,
    'Public Awareness Measures': 13,
    'Quarantine': 14,
    'Restriction and Regulation of Businesses': 15,
    'Restriction and Regulation of Government Services': 16,
    'Restrictions of Mass Gatherings':17,
    'Social Distancing':18
}

# Category mapping
df2['type_code'] = df2['type']
df2 = df2.replace({'type_code':category_codes})

df2.head()

#### Split train-test sets

In [ ]:
df2

In [ ]:
df = big_frame[["description", "type"]].dropna()
df = df.drop_duplicates(subset=["description"])

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['type'])
X_train, X_test, y_train, y_test = train_test_split(
    df['description'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)


In [ ]:
#TF Hub model
embed = hub.load("https://tfhub.dev/google/nnlm-en-dim128/2")

X_train_embed = embed(X_train.tolist())
X_test_embed = embed(X_test.tolist())

In [ ]:
# Build a tuned model
model = models.Sequential([
    layers.Input(shape=(128,), dtype=tf.float32),

    # First dense block
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    # Second dense block
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Output layer
    layers.Dense(len(label_encoder.classes_), activation='softmax')
])

# Compile with a lower initial learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True, verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1
)


history = model.fit(
    X_train_embed, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Evaluation

In [ ]:
y_pred = model.predict(X_test_embed)
y_pred_labels = tf.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_labels, target_names=label_encoder.classes_))